In [ ]:
# === Step 2: Manifest & Activity Check with Resume ===
import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from base64 import b64decode
from dotenv import load_dotenv
from time import sleep

# === Token Rotation ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
tokens = [t for t in tokens if t]
token_index = 0

def get_headers():
    global token_index
    headers = {"Authorization": f"token {tokens[token_index]}"}
    token_index = (token_index + 1) % len(tokens)
    return headers

# ✅ Define output folder
OUTPUT_DIR = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline"
os.makedirs(OUTPUT_DIR, exist_ok=True)

interim = os.path.join(OUTPUT_DIR, "step2_manifest_check_output.csv")
final_output = os.path.join(OUTPUT_DIR, "step2_manifest_final_output.csv")
step1_output = os.path.join(OUTPUT_DIR, "step1_url_lookup_output.csv")

try:
    df = pd.read_csv(interim)
except FileNotFoundError:
    df = pd.read_csv(step1_output)
    df = df.rename(columns={"valid_repo": "Valid_Repo_Step1"})  # Ensure correct naming
    df = df.assign(has_manifest="no", has_activity="no", nbr_of_manifest=-1)

# Process only rows where Step 1 repo was valid AND manifest hasn't been counted yet
target_df = df[(df["Valid_Repo_Step1"] == "yes") & (df["nbr_of_manifest"] == -1)]

for i, row in target_df.iterrows():
    html_url = row["html_url"]
    repo_path = html_url.replace("https://github.com/", "")
    print(f"{i}: {html_url}")
    r = requests.get(f"https://api.github.com/repos/{repo_path}/git/trees/HEAD?recursive=1", headers=get_headers())
    if r.status_code != 200:
        continue
    paths = [f['path'] for f in r.json().get("tree", []) if "AndroidManifest.xml" in f['path']]
    df.loc[df["html_url"] == html_url, "nbr_of_manifest"] = len(paths)
    if paths:
        df.loc[df["html_url"] == html_url, "has_manifest"] = "yes"
        for path in paths:
            f_url = f"https://api.github.com/repos/{repo_path}/contents/{path}"
            fr = requests.get(f_url, headers=get_headers())
            if fr.status_code != 200:
                continue
            try:
                content = b64decode(fr.json().get("content", "")).decode("utf-8", errors="ignore")
                if "<activity" in content:
                    df.loc[df["html_url"] == html_url, "has_activity"] = "yes"
                    break
            except Exception:
                continue

# Add validity flag for Step 2
df["Valid_Repo_Step2"] = df.apply(lambda r: "yes" if r["has_activity"] == "yes" else "no", axis=1)

# Save final output
df.to_csv(final_output, index=False)
print(f"✅ Step 2 complete. Output saved to {final_output}")


🔍 Checking: 000JustMe/PewCrypt
🔍 Checking: 000pp/Apepe
🔍 Checking: 0015/ThatProject
🔍 Checking: 008chen/InterpolatorShow
🔍 Checking: 00ec454/Ask
🔍 Checking: 00ec454/pop
🔍 Checking: 00-Evan/shattered-pixel-dungeon
🔍 Checking: 01101010110/proot-distro-scripts
🔍 Checking: 0-14N/NDroid
🔍 Checking: 029danio/fly
🔁 Rotated to token #2
🔍 Checking: 02cx/dong-rpc
🔍 Checking: 06peng/FrescoDemo
🔍 Checking: 08carmelo/android-keeplive
🔍 Checking: 0999312/Sakura_mod
🔍 Checking: 0ang3el/aem-rce-bundle
🔍 Checking: 0c34/govwa
🔍 Checking: 0Chencc/CTFCrackTools
🔍 Checking: 0Chencc/DaE
🔍 Checking: 0cyn/ktool
🔁 Rotated to token #3
🔍 Checking: 0dayCTF/Discord-Crash
🔍 Checking: 0ex/more-awesome
🔍 Checking: 0ffffffffh/dragondance
🔍 Checking: 0Kee-Team/JavaProbe
🔍 Checking: 0linlin0/CyberBox
🔍 Checking: 0linlin0/Java
🔍 Checking: 0linlin0/XPost
🔍 Checking: 0maru/twitter_login
🔁 Rotated to token #4
🔁 Rotated to token #5
🔁 Rotated to token #1
🔍 Checking: 0n1cOn3/FluxER
🔁 Rotated to token #2
🔁 Rotated to token #3
🔁